In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

FINAL_DIR = PROJECT_ROOT / "output" / "final"

print("Project root :", PROJECT_ROOT)
print("Final dir    :", FINAL_DIR)
print("Exists       :", FINAL_DIR.exists())

Project root : /Users/auliaaorama/Project/Ride hailing
Final dir    : /Users/auliaaorama/Project/Ride hailing/output/final
Exists       : True


In [2]:
final_files = {
    "monthly_quality": FINAL_DIR / "monthly_data_quality_2024_2026.csv",
    "business_pickup": FINAL_DIR / "business_pickup_2024_2026.parquet",
    "business_dropoff": FINAL_DIR / "business_dropoff_2024_2026.parquet",
    "hourly_demand": FINAL_DIR / "hourly_demand_2024_2026.parquet",
}

for name, path in final_files.items():
    size_mb = path.stat().st_size / (1024 ** 2)

    print(
        f"{name:20} "
        f"{size_mb:,.2f} MB "
        f"→ {path.name}"
    )

monthly_quality      0.01 MB → monthly_data_quality_2024_2026.csv
business_pickup      224.33 MB → business_pickup_2024_2026.parquet
business_dropoff     0.86 MB → business_dropoff_2024_2026.parquet
hourly_demand        13.06 MB → hourly_demand_2024_2026.parquet


### Upload stg_business_pickup

dilakukan dengan python karna lebih dari 100 mb

In [3]:
%pip install google-cloud-bigquery


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
from google.cloud import bigquery

print("BigQuery library berhasil di-import")

BigQuery library berhasil di-import


In [5]:
from google.cloud import bigquery
from pathlib import Path

PROJECT_ID = "ride-hailing-forecasting"
DATASET_ID = "ride_hailing"

client = bigquery.Client(project=PROJECT_ID)

print("Connected project:", client.project)

Connected project: ride-hailing-forecasting


In [6]:
dataset_id = f"{PROJECT_ID}.{DATASET_ID}"

dataset = client.get_dataset(dataset_id)

print("Dataset :", dataset.dataset_id)
print("Location:", dataset.location)

Dataset : ride_hailing
Location: US


In [7]:
business_pickup_file = (
    FINAL_DIR / "business_pickup_2024_2026.parquet"
)

print("File exists:", business_pickup_file.exists())
print(
    "File size:",
    round(
        business_pickup_file.stat().st_size / (1024 ** 2),
        2
    ),
    "MB"
)

File exists: True
File size: 224.33 MB


In [8]:
table_id = (
    f"{PROJECT_ID}."
    f"{DATASET_ID}."
    f"stg_business_pickup"
)

print(table_id)

ride-hailing-forecasting.ride_hailing.stg_business_pickup


In [9]:
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

In [10]:
with open(business_pickup_file, "rb") as source_file:

    load_job = client.load_table_from_file(
        source_file,
        table_id,
        job_config=job_config,
        location=dataset.location
    )

print("Upload dimulai...")
print("Job ID:", load_job.job_id)

Upload dimulai...
Job ID: c470953f-5dad-4572-a101-b81db745b235


In [11]:
load_job.result()

print("Upload selesai.")

Upload selesai.


In [12]:
table = client.get_table(table_id)

print("Table :", table.table_id)
print("Rows  :", f"{table.num_rows:,}")
print("Cols  :", len(table.schema))

Table : stg_business_pickup
Rows  : 10,446,585
Cols  : 17
